# Settings

## Import Packages

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency, pointbiserialr

import json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import math
import itertools
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import Point
from shapely.ops import unary_union
import requests
import time
from geopy.geocoders import Nominatim
import time
import geopandas as gpd
from shapely.geometry import box
from shapely.wkt import dumps
from shapely.wkt import loads


pd.set_option('display.precision', 4)

## Self-defined functions

In [2]:
def save_geoDataFrame(source_df, filename):
    """
    Save a GeoDataFrame to a CSV file by converting the geometry column to WKT format.

    Parameters:
        source_df (GeoDataFrame): The GeoDataFrame to be saved.
        filename (str): Path to the output CSV file.

    Notes:
        - The geometry column will be converted to WKT strings using shapely.wkt.dumps().
        - The resulting CSV can be restored into a GeoDataFrame using shapely.wkt.loads().
    """
    df = source_df.copy()
    df['geometry'] = df['geometry'].apply(dumps)
    df.to_csv(filename, index=False)

In [3]:
def load_geoDataFrame(filepath):
    """
    Load a GeoDataFrame from a CSV file with a geometry column in WKT format.

    Parameters:
        filepath (str): Path to the CSV file containing the saved GeoDataFrame.

    Returns:
        GeoDataFrame: A GeoDataFrame reconstructed from the CSV, with geometries parsed from WKT strings.
    
    Notes:
        - Assumes the geometry column is named 'geometry' and stored in WKT format.
        - The returned GeoDataFrame is assigned the CRS 'EPSG:4326' by default.
    """
    df = pd.read_csv(filepath)
    df['geometry'] = df['geometry'].apply(loads)
    return gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

# Manhattan Grid
- Generate Manhattan grid based ON NYC TLC Taxi Zones
- Source: [NYC TLC Taxi Zone Shapefile](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)
- Instructions:
	1.	Download the **Taxi Zone Shapefile** from the link above.
	2.	Unzip the contents.
	3.	Move the extracted `taxi_zones` folder into the `raw_data/` directory of your project.

In [4]:
# Read the NYC taxi zone shapefile (typically in EPSG:2263 projection)
taxi_zones = gpd.read_file('raw_data/taxi_zones/taxi_zones.shp')

# Reproject to EPSG:2263 (a local projected CRS suitable for geometric operations)
projected = taxi_zones.to_crs(epsg=2263)
projected['centroid'] = projected.geometry.centroid

# Convert the centroids back to geographic coordinates (EPSG:4326, i.e. latitude/longitude)
centroids = projected.set_geometry('centroid').to_crs(epsg=4326)
taxi_zones['longitude'] = centroids.geometry.x
taxi_zones['latitude'] = centroids.geometry.y

# Only keep zones of Manhattan borough
manhattan_zones = taxi_zones[taxi_zones['borough'] == 'Manhattan']
manhattan_zones = manhattan_zones[manhattan_zones.is_valid]
manhattan_zones['geometry'] = manhattan_zones['geometry'].buffer(0)  # Fix geometry issues
manhattan_zones = manhattan_zones.to_crs(epsg=4326)  # Ensure CRS is WGS84

# Save the result as CSV file
manhattan_zones = manhattan_zones.rename(columns={
    'OBJECTID': 'object_id',
    'Shape_Leng': 'shape_length',
    'Shape_Area': 'shape_area',
    'zone': 'zone',
    'LocationID': 'location_id',
    'borough': 'borough',
    'geometry': 'geometry',
    'longitude': 'longitude',
    'latitude': 'latitude'
})
manhattan_zones.head()

,object_id,shape_length,shape_area,zone,location_id,borough,geometry,longitude,latitude
3,4,0.0436,1.1187e-04,Alphabet City,4,Manhattan,"POLYGON ((-73.97177 40.72582, -73.97179 40.725...",-73.9770,40.7238
11,12,0.0367,4.1512e-05,Battery Park,12,Manhattan,"POLYGON ((-74.01566 40.70483, -74.0154 40.7048...",-74.0156,40.7029
12,13,0.0503,1.4936e-04,Battery Park City,13,Manhattan,"POLYGON ((-74.01244 40.71906, -74.01282 40.717...",-74.0161,40.7120
23,24,0.0470,6.0724e-05,Bloomingdale,24,Manhattan,"POLYGON ((-73.95954 40.79872, -73.96004 40.798...",-73.9655,40.8020
40,41,0.0528,1.4309e-04,Central Harlem,41,Manhattan,"POLYGON ((-73.94774 40.8096, -73.94506 40.8084...",-73.9513,40.8043


In [10]:
save_geoDataFrame(source_df=manhattan_zones,
                  filename='taxi_zones.csv')

/var/folders/l_/zlzp_vvs4j31hx3pg4j26tch0000gn/T/ipykernel_66044/1955781453.py:14: UserWarning: Geometry column does not contain geometry.
  df['geometry'] = df['geometry'].apply(dumps)


## Generate 250m x 250m Manhattan Grid

In [5]:
# Union all geometries to form one Manhattan shape
manhattan_union = unary_union(manhattan_zones.geometry)
minx, miny, maxx, maxy = manhattan_union.bounds

# Approximate degree step based on meters (note: latitude is ~111km/deg, longitude ~85km/deg at NYC)
# Use 250m as grid size

grid_size = 250

step_lat = grid_size / 111000
step_lon = grid_size / 85000

# Create grid cells
lat_points = np.arange(miny, maxy, step_lat)
lon_points = np.arange(minx, maxx, step_lon)
grid_cells = [box(lon, lat, lon + step_lon, lat + step_lat)
              for lat in lat_points for lon in lon_points]

grid = gpd.GeoDataFrame(geometry=grid_cells, crs="EPSG:4326")

# Clip grid to Manhattan shape
manhattan_grid = grid[grid.intersects(manhattan_union)].copy()

# Project to EPSG:2263 (NYC local CRS) for accurate centroid calculation
grid_projected = manhattan_grid.to_crs(epsg=2263)
grid_projected['centroid'] = grid_projected.geometry.centroid

# Transform centroids back to WGS84
centroids = gpd.GeoSeries(grid_projected['centroid'], crs='EPSG:2263').to_crs(epsg=4326)

# Attach centroid coordinates and grid ID
manhattan_grid['lat'] = centroids.y
manhattan_grid['lon'] = centroids.x
manhattan_grid = manhattan_grid.reset_index(drop=True)
manhattan_grid['grid_id'] = manhattan_grid.index.map(lambda i: f"M-{i+1:04d}")
manhattan_grid = manhattan_grid[['grid_id', 'lat', 'lon', 'geometry']]

In [6]:
save_geoDataFrame(source_df=manhattan_grid,
                  filename='manhattan_grid.csv')

/var/folders/l_/zlzp_vvs4j31hx3pg4j26tch0000gn/T/ipykernel_66044/1955781453.py:14: UserWarning: Geometry column does not contain geometry.
  df['geometry'] = df['geometry'].apply(dumps)


In [7]:
manhattan_grid = load_geoDataFrame('manhattan_grid.csv')
manhattan_grid.head()

,grid_id,lat,lon,geometry
0,M-0001,40.6840,-74.0257,"POLYGON ((-74.0242 40.68292, -74.0242 40.68517..."
1,M-0002,40.6840,-74.0227,"POLYGON ((-74.02126 40.68292, -74.02126 40.685..."
2,M-0003,40.6840,-74.0198,"POLYGON ((-74.01832 40.68292, -74.01832 40.685..."
3,M-0004,40.6863,-74.0257,"POLYGON ((-74.0242 40.68517, -74.0242 40.68742..."
4,M-0005,40.6863,-74.0227,"POLYGON ((-74.02126 40.68517, -74.02126 40.687..."


## Map Taxi Location ID to Manhattan Grid

In [8]:
manhattan_zones = manhattan_zones.to_crs(epsg=4326)
zone_to_grid = gpd.sjoin(manhattan_zones, manhattan_grid[['grid_id', 'geometry']], how='left', predicate='intersects')
zone_to_grid = zone_to_grid[['location_id', 'geometry', 'longitude', 'latitude', 'zone', 'shape_length', 'shape_area', 'grid_id']]
zone_to_grid = zone_to_grid.sort_values(['location_id', 'grid_id'])
zone_to_grid.head()

,location_id,geometry,longitude,latitude,zone,shape_length,shape_area,grid_id
3,4,"POLYGON ((-73.97177 40.72582, -73.97179 40.725...",-73.977,40.7238,Alphabet City,0.0436,0.0001,M-0132
3,4,"POLYGON ((-73.97177 40.72582, -73.97179 40.725...",-73.977,40.7238,Alphabet City,0.0436,0.0001,M-0133
3,4,"POLYGON ((-73.97177 40.72582, -73.97179 40.725...",-73.977,40.7238,Alphabet City,0.0436,0.0001,M-0145
3,4,"POLYGON ((-73.97177 40.72582, -73.97179 40.725...",-73.977,40.7238,Alphabet City,0.0436,0.0001,M-0146
3,4,"POLYGON ((-73.97177 40.72582, -73.97179 40.725...",-73.977,40.7238,Alphabet City,0.0436,0.0001,M-0147


In [9]:
save_geoDataFrame(source_df=zone_to_grid,
                  filename='taxi_location_grid_mapping.csv')

/var/folders/l_/zlzp_vvs4j31hx3pg4j26tch0000gn/T/ipykernel_66044/1955781453.py:14: UserWarning: Geometry column does not contain geometry.
  df['geometry'] = df['geometry'].apply(dumps)
